<a href="https://colab.research.google.com/github/TorbenGl/visionseminar/blob/main/notebooks/01_computer_vision_and_mlp_mnist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Learning for Maritime Vision Applications
## Session 0 · Part 2 — Computer Vision & Your First Image Classifier
*Torben Globisch · VACOT*

---

| Notebook | Topic |
|---|---|
| 00 | Artificial Neural Networks: neurons, layers, learning |
| **→ 01** | **Computer Vision & an MLP on MNIST** |
| 02 | Convolutional Neural Networks |
| 03 | Vision Foundation Models: the CNN era |
| 04 | Vision Transformers |
| 05 | Self-supervised pretraining & transfer |

**By the end of this notebook you can**
1. explain how a computer "sees" an image (spoiler: as a big array of numbers),
2. name the core computer-vision tasks and what each one outputs,
3. train a fully-connected network (MLP) on handwritten digits (**Exercise 2**),
4. show with experiments why an MLP is the wrong tool for images:
   * it is **fragile**: moving a digit by 2–3 pixels breaks it,
   * it **ignores the spatial structure**: shuffling the pixels doesn't bother it,
   * it **does not scale**: at 224 × 224 × 3 it becomes huge, slow, and finally does not fit into memory at all.

> 💡 Use a GPU runtime: *Runtime → Change runtime type → T4 GPU*.

In [ ]:
# @title ⚙️ Setup: imports, colours & helpers — run me first { display-mode: "form" }
import math, io, re, base64, time, urllib.request
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.patches import Rectangle, FancyBboxPatch, Circle, FancyArrowPatch
from IPython.display import HTML, display
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F

# Colour palette — the same colours as the lecture slides
NAVY, BLUE, LAVENDER, ORANGE, GREEN, RED, GREY, LIGHT = (
    "#2D3A8C", "#5B67B8", "#A3A9DB", "#E8702A", "#3E9E64", "#C0392B", "#8A8FA8", "#EEF0F7")
plt.rcParams.update({
    "figure.dpi": 100, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titleweight": "bold", "axes.titlesize": 11, "axes.titlecolor": NAVY,
    "axes.prop_cycle": mpl.cycler(color=[NAVY, ORANGE, GREEN, BLUE, RED, GREY]),
    "animation.html": "jshtml", "animation.embed_limit": 80,
})

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0); np.random.seed(0)
print(f"PyTorch {torch.__version__} | device: {device}")
if device.type == "cpu":
    print("⚠️  No GPU found. In Colab: Runtime → Change runtime type → T4 GPU")

def play(anim):
    """Show a matplotlib animation as an HTML player (play / pause / step frame by frame)."""
    plt.close(anim._fig)
    def shrink(m):  # re-encode every frame as a 256-colour PNG: 3-4x smaller, looks the same
        img = Image.open(io.BytesIO(base64.b64decode(m.group(1).replace("\\\n", ""))))
        buf = io.BytesIO(); img.convert("RGB").quantize(256).save(buf, "png", optimize=True)
        return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()
    return HTML(re.sub(r"data:image/png;base64,([A-Za-z0-9+/=\\\n]+)", shrink, anim.to_jshtml()))

# Maritime example photos from the COCO validation set (Flickr, Creative Commons licences).
MARITIME = {"trawler": 330369, "sailboat": 286422, "lifeboat": 400803, "marina": 442822,
            "fishing_boat": 451090, "yacht": 489014, "ferry": 543300}

def load_image(name, size=None):
    """Load one of the MARITIME photos (or any image URL) as an RGB PIL image."""
    url = f"http://images.cocodataset.org/val2017/{MARITIME[name]:012d}.jpg" if name in MARITIME else name
    with urllib.request.urlopen(url, timeout=30) as r:
        img = Image.open(io.BytesIO(r.read())).convert("RGB")
    if size is not None:  # centre-crop to a square, then resize
        s = min(img.size); l, t = (img.width - s) // 2, (img.height - s) // 2
        img = img.crop((l, t, l + s, t + s)).resize((size, size), Image.BICUBIC)
    return img

---
# 1 · What is Computer Vision?

> **Computer Vision** enables machines to interpret and understand visual information (images, videos and 3D data).

Applications: autonomous driving · medical imaging · satellite analysis · quality control · **maritime monitoring**
(vessel detection, collision avoidance, debris & oil-spill detection, underwater inspection, …)

## 1.1 · The machine only sees numbers
Here is a fishing trawler. To the computer it is a $H \times W \times 3$ array of integers in $[0, 255]$: one value per pixel
for each of the red, green and blue channels.

In [ ]:
img = load_image("trawler")
arr = np.array(img)
print("shape:", arr.shape, " dtype:", arr.dtype, f" → {arr.size:,} numbers")

# zoom into a small patch and print its values
y0, x0, s = 274, 159, 8   # a patch on the border between the white stripe and the red hull
patch = arr[y0:y0 + s, x0:x0 + s]

fig = plt.figure(figsize=(15, 4.6))
gs = fig.add_gridspec(1, 3, width_ratios=[1.5, 1, 1.15])
ax = fig.add_subplot(gs[0]); ax.imshow(arr); ax.set_title(f"What we see  ({arr.shape[1]}×{arr.shape[0]} px)")
ax.add_patch(Rectangle((x0, y0), s, s, fill=False, ec=ORANGE, lw=2.5)); ax.axis("off")
ax = fig.add_subplot(gs[1]); ax.imshow(patch, interpolation="nearest"); ax.set_title(f"zoom: {s}×{s} patch"); ax.axis("off")
ax = fig.add_subplot(gs[2]); gray = patch.mean(2)
ax.imshow(gray, cmap="gray", vmin=0, vmax=255)
for i in range(s):
    for j in range(s):
        ax.text(j, i, int(gray[i, j]), ha="center", va="center", fontsize=8, color="white" if gray[i, j] < 128 else "black")
ax.set_title("What the computer sees (grey values)"); ax.axis("off")
plt.tight_layout(); plt.show()

A colour image is three such grids stacked on top of each other:

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3))
axes[0].imshow(arr); axes[0].set_title("RGB image")
for ax, c, name, cmap in zip(axes[1:], range(3), ["Red", "Green", "Blue"], ["Reds", "Greens", "Blues"]):
    ax.imshow(arr[..., c], cmap=cmap); ax.set_title(f"{name} channel")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()
print(f"A standard 224 × 224 colour input already has 224·224·3 = {224 * 224 * 3:,} numbers.")

## 1.2 · Why is vision hard?
The same boat looks completely different under a different viewpoint, lighting or scale, and the numbers change completely too.
Let's measure how "far apart" images are in raw pixel space:

In [ ]:
base = np.asarray(load_image("trawler", size=224), dtype=np.float32) / 255
variants = {
    "shifted 10 px": np.roll(base, 10, axis=1),
    "darker (×0.6)": base * 0.6,
    "mirrored": base[:, ::-1],
    "a different boat": np.asarray(load_image("lifeboat", size=224), dtype=np.float32) / 255,
    "random noise": np.random.rand(*base.shape).astype(np.float32),
}
dist = {k: np.sqrt(((v - base) ** 2).mean()) for k, v in variants.items()}

fig, axes = plt.subplots(1, 6, figsize=(16, 3.3))
axes[0].imshow(base); axes[0].set_title("original")
for ax, (k, v) in zip(axes[1:], variants.items()):
    ax.imshow(np.clip(v, 0, 1)); ax.set_title(f"{k}\npixel distance = {dist[k]:.2f}", fontsize=10)
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

Shifting the *same* trawler by a few pixels moves it about as far away in pixel space as a picture of a **different boat**!
Raw pixel values are a terrible description of *what* is in an image.

> **Key insight:** useful visual features (edges, textures, shapes, parts) must be **learned from data**, not hand-crafted.

---
# 2 · Core Computer Vision Tasks

| Task | Question | Output | Maritime example |
|---|---|---|---|
| **Classification** | *What* is in the image? | one label per image | ship / no ship |
| **Object detection** | *Where* is each object? | bounding boxes + classes | locate vessels in a harbour |
| **Segmentation** | *Which pixel* belongs to what? | a label per pixel | water vs. hull vs. debris |
| **Depth estimation** | *How far* away is it? | a distance per pixel | 3D maps of a scene or the seafloor |

Let's run all four on maritime photos. We use off-the-shelf **pretrained** models. You will learn how they work
over the next sessions; for now we only look at **what** they output.

*(This downloads ~150 MB of model weights; it takes about a minute.)*

In [ ]:
import torchvision
from torchvision.models import ResNet50_Weights
from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_fpn, FasterRCNN_MobileNet_V3_Large_FPN_Weights
from torchvision.models.segmentation import deeplabv3_mobilenet_v3_large, DeepLabV3_MobileNet_V3_Large_Weights
from torchvision.transforms.functional import to_tensor
from transformers import pipeline

img_cls, img_det, img_seg = load_image("trawler"), load_image("marina"), load_image("fishing_boat")

with torch.no_grad():
    # 1) Classification: ResNet-50 trained on ImageNet (1000 classes)
    w = ResNet50_Weights.DEFAULT
    clf = torchvision.models.resnet50(weights=w).eval().to(device)
    probs = clf(w.transforms()(img_cls)[None].to(device)).softmax(1)[0].cpu()
    top = probs.topk(5)

    # 2) Object detection: Faster R-CNN trained on COCO (80 classes)
    w = FasterRCNN_MobileNet_V3_Large_FPN_Weights.DEFAULT
    det = fasterrcnn_mobilenet_v3_large_fpn(weights=w, box_score_thresh=0.75).eval().to(device)
    det_out = det([to_tensor(img_det).to(device)])[0]
    det_names = w.meta["categories"]

    # 3) Semantic segmentation: DeepLabv3 trained on Pascal VOC classes (21 classes, incl. "boat")
    w = DeepLabV3_MobileNet_V3_Large_Weights.DEFAULT
    seg = deeplabv3_mobilenet_v3_large(weights=w).eval().to(device)
    seg_mask = seg(w.transforms()(img_seg)[None].to(device))["out"].argmax(1)[0].cpu()
    seg_names = w.meta["categories"]

# 4) Monocular depth estimation: Depth Anything V2 (small)
depth = pipeline("depth-estimation", model="depth-anything/Depth-Anything-V2-Small-hf", device=0 if device.type == "cuda" else -1)
depth_map = depth(img_seg)["depth"]

fig, axes = plt.subplots(1, 4, figsize=(18, 4.4))
ax = axes[0]; ax.imshow(img_cls); ax.axis("off"); ax.set_title("Classification")
txt = "\n".join(f"{ResNet50_Weights.DEFAULT.meta['categories'][i]:<16s}{p:6.1%}" for p, i in zip(top.values, top.indices))
ax.text(0.02, 0.02, txt, transform=ax.transAxes, family="monospace", fontsize=9, va="bottom", bbox=dict(fc="white", alpha=.85, ec=NAVY))

ax = axes[1]; ax.imshow(img_det); ax.axis("off"); ax.set_title(f"Object detection ({len(det_out['boxes'])} objects)")
for box, lbl, sc in zip(det_out["boxes"].cpu(), det_out["labels"].cpu(), det_out["scores"].cpu()):
    x1, y1, x2, y2 = box
    ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec=ORANGE, lw=2))
    ax.text(x1, y1 - 3, f"{det_names[lbl]} {sc:.0%}", color="white", fontsize=7, bbox=dict(fc=ORANGE, ec="none", pad=1))

ax = axes[2]; seg_small = img_seg.resize(seg_mask.shape[::-1])
ax.imshow(seg_small); ax.axis("off")
mask_rgba = np.zeros((*seg_mask.shape, 4)); mask_rgba[seg_mask == seg_names.index("boat")] = mpl.colors.to_rgba(ORANGE, 0.65)
ax.imshow(mask_rgba); ax.set_title("Semantic segmentation (boat pixels)")

ax = axes[3]; ax.imshow(depth_map, cmap="magma"); ax.axis("off"); ax.set_title("Depth estimation (bright = near)")
plt.tight_layout(); plt.show()

Note that the classifier has never heard of a "trawler": ImageNet's 1000 classes contain *fireboat*, *lifeboat*,
*container ship*, … but no fishing vessels. A model can only predict the labels it was trained on, which is a
common problem when we take general-purpose models to the maritime domain.

> **Key observation:** all four tasks share the same core challenge: extracting meaningful **spatial features** from raw
> pixel arrays. The backbone networks that do this are the topic of the rest of the course.

Today we start with the simplest task, **classification**, on the simplest dataset.

---
# ✏️ Exercise 2: Classify handwritten digits with an MLP

**Dataset: MNIST**
* 60 000 training + 10 000 test images
* 28 × 28 grayscale, digits 0–9

**Tasks**
1. flatten each image: 28 × 28 → 784 inputs
2. build an MLP: 784 → 256 → 128 → 10
3. train with cross-entropy
4. plot the training loss per epoch
5. visualise example predictions

🎯 **Goal:** test accuracy > 95 %

## Step 0: load and look at the data

In [ ]:
from torchvision import datasets
from torch.utils.data import TensorDataset, DataLoader

mnist_train = datasets.MNIST("data", train=True, download=True)
mnist_test = datasets.MNIST("data", train=False, download=True)

# Keep everything as tensors: images in [0, 1], then standardise with the usual MNIST mean/std
MEAN, STD = 0.1307, 0.3081
X_train = ((mnist_train.data.float() / 255) - MEAN) / STD   # (60000, 28, 28)
X_test = ((mnist_test.data.float() / 255) - MEAN) / STD     # (10000, 28, 28)
y_train, y_test = mnist_train.targets, mnist_test.targets
print("train:", tuple(X_train.shape), " test:", tuple(X_test.shape))

fig, axes = plt.subplots(3, 10, figsize=(12, 3.8))
for d in range(10):
    idx = (y_train == d).nonzero()[:3, 0]
    for r, i in enumerate(idx):
        axes[r, d].imshow(mnist_train.data[i], cmap="gray_r"); axes[r, d].axis("off")
    axes[0, d].set_title(str(d), fontsize=13)
plt.suptitle("MNIST: 3 examples per class", color=NAVY, fontweight="bold"); plt.tight_layout(); plt.show()

## Step 1: flatten
An MLP takes a **vector** as input, so each 28 × 28 image becomes one long vector of 784 numbers, row after row.

In [ ]:
digit = mnist_train.data[0].numpy()
fig = plt.figure(figsize=(15, 3.6))
gs = fig.add_gridspec(2, 2, width_ratios=[1, 5], height_ratios=[4, 1])
ax = fig.add_subplot(gs[:, 0]); ax.imshow(digit, cmap="gray_r")
for r in [5, 6, 7]: ax.add_patch(Rectangle((-0.5, r - 0.5), 28, 1, fill=False, ec=ORANGE, lw=1.2))
ax.set_title(f"28 × 28 image (label {y_train[0].item()})"); ax.axis("off")
ax = fig.add_subplot(gs[0, 1]); ax.imshow(digit.reshape(1, -1), cmap="gray_r", aspect="auto")
ax.set_yticks([]); ax.set_title("flattened → one vector of 784 numbers (rows 5, 6, 7 marked)")
for r in [5, 6, 7]: ax.axvspan(r * 28 - 0.5, (r + 1) * 28 - 0.5, fc="none", ec=ORANGE, lw=1.2)
ax = fig.add_subplot(gs[1, 1]); ax.axis("off")
ax.text(0.5, 0.5, "Flatten first! The MLP no longer knows which pixels were neighbours.", ha="center", va="center",
        color=NAVY, fontweight="bold", fontsize=12, transform=ax.transAxes)
plt.tight_layout(); plt.show()

## A reusable training helper
The loop is exactly the one from notebook 00, now with mini-batches. We will reuse `fit` in the next notebooks.

In [ ]:
def make_loaders(X_tr, y_tr, X_te, y_te, batch_size=128):
    train = DataLoader(TensorDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True)
    test = DataLoader(TensorDataset(X_te, y_te), batch_size=1000)
    return train, test

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss += F.cross_entropy(logits, y, reduction="sum").item()
        correct += (logits.argmax(1) == y).sum().item(); n += len(y)
    return loss / n, correct / n

def fit(model, train_loader, test_loader, epochs=5, lr=1e-3):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    hist = {"train_loss": [], "test_loss": [], "test_acc": [], "step_loss": []}
    for epoch in range(epochs):
        model.train(); t0, running = time.time(), 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            loss = F.cross_entropy(model(x), y)      # forward + loss
            opt.zero_grad(); loss.backward(); opt.step()   # backprop + update
            running += loss.item() * len(y); hist["step_loss"].append(loss.item())
        test_loss, test_acc = evaluate(model, test_loader)
        hist["train_loss"].append(running / len(train_loader.dataset))
        hist["test_loss"].append(test_loss); hist["test_acc"].append(test_acc)
        print(f"epoch {epoch + 1}/{epochs}  train loss {hist['train_loss'][-1]:.4f}  "
              f"test loss {test_loss:.4f}  test acc {test_acc:.2%}  ({time.time() - t0:.1f}s)")
    return hist

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

train_loader, test_loader = make_loaders(X_train, y_train, X_test, y_test)

## ✏️ Step 2: build the MLP
Use `nn.Flatten()` as the first layer, then **784 → 256 → 128 → 10** with ReLU activations.

In [ ]:
def build_mlp():
    # TODO: return nn.Sequential(nn.Flatten(), ...)
    return None

In [ ]:
# @title 💡 Solution: Step 2 { display-mode: "form" }
def build_mlp():
    return nn.Sequential(
        nn.Flatten(),                       # (B, 28, 28) → (B, 784)
        nn.Linear(784, 256), nn.ReLU(),
        nn.Linear(256, 128), nn.ReLU(),
        nn.Linear(128, 10),                 # one logit per digit
    )

In [ ]:
mlp = build_mlp()
print(mlp)
print(f"\nparameters: {count_params(mlp):,}   (784·256+256) + (256·128+128) + (128·10+10) = "
      f"{(784*256+256) + (256*128+128) + (128*10+10):,}")

## Step 3: train

In [ ]:
torch.manual_seed(0)
mlp = build_mlp()
hist_mlp = fit(mlp, train_loader, test_loader, epochs=5)

## Step 4: plot the training curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
axes[0].plot(hist_mlp["step_loss"], color=LAVENDER, lw=.6, label="per mini-batch")
k = 50; smooth = np.convolve(hist_mlp["step_loss"], np.ones(k) / k, mode="valid")
axes[0].plot(range(k - 1, len(hist_mlp["step_loss"])), smooth, color=NAVY, lw=2, label=f"moving avg ({k})")
axes[0].set(xlabel="training step", ylabel="loss", title="Loss per mini-batch", yscale="log"); axes[0].legend()
ep = range(1, len(hist_mlp["train_loss"]) + 1)
axes[1].plot(ep, hist_mlp["train_loss"], "-o", label="train"); axes[1].plot(ep, hist_mlp["test_loss"], "-o", label="test")
axes[1].set(xlabel="epoch", ylabel="loss", title="Loss per epoch"); axes[1].legend()
axes[2].plot(ep, hist_mlp["test_acc"], "-o", color=GREEN); axes[2].axhline(0.95, color=RED, ls="--", lw=1, label="goal 95 %")
axes[2].set(xlabel="epoch", ylabel="accuracy", title="Test accuracy"); axes[2].legend()
plt.tight_layout(); plt.show()

## Step 5: look at the predictions
Accuracy alone hides a lot. Look at individual predictions, the confusion matrix and the **mistakes**.

In [ ]:
@torch.no_grad()
def predict(model, X):
    model.eval()
    return torch.cat([model(X[i:i + 1000].to(device)).softmax(1).cpu() for i in range(0, len(X), 1000)])

probs_mlp = predict(mlp, X_test); pred_mlp = probs_mlp.argmax(1)

fig, axes = plt.subplots(2, 10, figsize=(15, 3.6))
for k, ax in enumerate(axes[0]):
    ax.imshow(mnist_test.data[k], cmap="gray_r"); ax.axis("off")
    ok = pred_mlp[k] == y_test[k]
    ax.set_title(f"pred {pred_mlp[k].item()}\n{probs_mlp[k].max():.0%}", color=GREEN if ok else RED, fontsize=10)
wrong = (pred_mlp != y_test).nonzero()[:, 0]
for ax, k in zip(axes[1], wrong[:10]):
    ax.imshow(mnist_test.data[k], cmap="gray_r"); ax.axis("off")
    ax.set_title(f"pred {pred_mlp[k].item()} / true {y_test[k].item()}", color=RED, fontsize=9)
axes[0, 0].text(-12, 14, "first 10\ntest images", fontsize=10, ha="right", va="center", color=NAVY)
axes[1, 0].text(-12, 14, "mistakes", fontsize=10, ha="right", va="center", color=RED)
plt.tight_layout(); plt.show()

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from matplotlib.colors import LinearSegmentedColormap, LogNorm
cm = confusion_matrix(y_test, pred_mlp)
fig, ax = plt.subplots(figsize=(5.5, 5))
ConfusionMatrixDisplay(cm).plot(ax=ax, cmap=LinearSegmentedColormap.from_list("navy", ["white", NAVY]), colorbar=False,
                                im_kw=dict(norm=LogNorm(vmin=1, vmax=cm.max())))
ax.set_title(f"Confusion matrix (log colour scale)\n{len(wrong)} mistakes out of {len(y_test):,}"); plt.show()

---
# 3 · How robust is the MLP? Move the digit by a few pixels

97–98 % sounds great. But what has the MLP actually learned? Every neuron in the first hidden layer has **one weight per
pixel position**. Let's look at the 784 weights of a few neurons, drawn as 28 × 28 images:

In [ ]:
W_first = mlp[1].weight.detach().cpu().reshape(-1, 28, 28)      # (256, 28, 28): one weight image per hidden neuron
fig, axes = plt.subplots(2, 8, figsize=(13, 3.6))
for k, ax in enumerate(axes.ravel()):
    lim = W_first[k].abs().max()
    ax.imshow(W_first[k], cmap="coolwarm", vmin=-lim, vmax=lim); ax.axis("off")
plt.suptitle("First-layer weights of 16 hidden neurons  (red = positive, blue = negative)", color=NAVY, fontweight="bold")
plt.tight_layout(); plt.show()

Each neuron is a template **glued to fixed pixel positions**. A stroke in the centre and the same stroke two pixels to the
right hit completely different weights. MNIST digits are always centred in their 28 × 28 box, so the MLP never had to deal
with anything else. What happens if we move the test digits a little?

In [ ]:
BACKGROUND = (0 - MEAN) / STD                    # the value of an empty pixel after standardising

def shift(X, dx, dy):
    """Move images dx pixels to the right and dy pixels down; the uncovered border becomes background."""
    P = 8
    Xp = F.pad(X, (P, P, P, P), value=BACKGROUND)
    return Xp[:, P - dy:P - dy + 28, P - dx:P - dx + 28]

@torch.no_grad()
def accuracy(model, X, y):
    model.eval()
    pred = torch.cat([model(X[i:i + 2000].to(device)).argmax(1).cpu() for i in range(0, len(X), 2000)])
    return (pred == y).float().mean().item()

SHIFTS = list(range(-4, 5))
acc_mlp_shift = np.array([[accuracy(mlp, shift(X_test, dx, dy), y_test) for dx in SHIFTS] for dy in SHIFTS])
print("accuracy when moving all test digits sideways (dy = 0):")
print("  ".join(f"{dx:+d} px: {a:.0%}" for dx, a in zip(SHIFTS, acc_mlp_shift[SHIFTS.index(0)])))

In [ ]:
# @title 🎨 Figure: shifted digits and the accuracy for every shift { display-mode: "form" }
with torch.no_grad():   # two test digits the MLP gets right (confidently) in place, but wrong after moving them 2 px
    p0 = mlp(X_test[:500].to(device)).softmax(1).cpu()
    ok0 = (p0.argmax(1) == y_test[:500]) & (p0.max(1).values > 0.95)
    ok2 = mlp(shift(X_test[:500], 2, 0).to(device)).argmax(1).cpu() == y_test[:500]
EXAMPLES = (ok0 & ~ok2).nonzero()[:2, 0].tolist()

fig = plt.figure(figsize=(15.5, 4.6))
gs = fig.add_gridspec(2, 7, width_ratios=[1, 1, 1, 1, 1, 0.45, 3.2], hspace=0.35)
for r, k in enumerate(EXAMPLES):
    for c, dx in enumerate([0, 1, 2, 3, 4]):
        ax = fig.add_subplot(gs[r, c]); xs = shift(X_test[k:k + 1], dx, 0)
        with torch.no_grad(): p = mlp(xs.to(device)).softmax(1)[0].cpu()
        ok = p.argmax().item() == y_test[k].item()
        ax.imshow(xs[0], cmap="gray_r"); ax.set_xticks([]); ax.set_yticks([])
        for sp in ax.spines.values(): sp.set_visible(True); sp.set_color(LAVENDER)
        ax.set_title(f"+{dx} px → {p.argmax().item()} ({p.max():.0%})", color=GREEN if ok else RED, fontsize=10)
ax = fig.add_subplot(gs[:, 6])
im = ax.imshow(acc_mlp_shift, cmap="RdYlGn", vmin=0, vmax=1)
for (r, c), v in np.ndenumerate(acc_mlp_shift): ax.text(c, r, f"{v:.0%}", ha="center", va="center", fontsize=8)
ax.set_xticks(range(len(SHIFTS)), [f"{s:+d}" for s in SHIFTS]); ax.set_yticks(range(len(SHIFTS)), [f"{s:+d}" for s in SHIFTS])
ax.set(xlabel="shift right (px)", ylabel="shift down (px)", title="MLP test accuracy after moving every digit")
plt.colorbar(im, ax=ax, shrink=.85); plt.show()

In [ ]:
# @title 🎞️ Animation: a digit slides sideways, the MLP changes its mind { display-mode: "form" }
K_DIGIT = EXAMPLES[0]
dxs = list(range(0, 7)) + list(range(5, -7, -1)) + list(range(-5, 1))
with torch.no_grad():
    frames_x = [shift(X_test[K_DIGIT:K_DIGIT + 1], dx, 0) for dx in dxs]
    frames_p = [mlp(x.to(device)).softmax(1)[0].cpu().numpy() for x in frames_x]
true = y_test[K_DIGIT].item()
curve_dx = np.arange(-6, 7)
curve_acc = [accuracy(mlp, shift(X_test[:2000], dx, 0), y_test[:2000]) for dx in curve_dx]

fig, axes = plt.subplots(1, 3, figsize=(15, 3.9), gridspec_kw=dict(width_ratios=[0.9, 1.4, 1.3]))
im = axes[0].imshow(frames_x[0][0], cmap="gray_r", vmin=X_test.min(), vmax=X_test.max()); axes[0].set_xticks([]); axes[0].set_yticks([])
for sp in axes[0].spines.values(): sp.set_visible(True); sp.set_color(LAVENDER)
bars = axes[1].bar(range(10), frames_p[0], color=GREY)
axes[1].set(xticks=range(10), ylim=(0, 1.05), xlabel="digit class", ylabel="probability", title="MLP output")
axes[2].plot(curve_dx, curve_acc, "-o", color=NAVY, ms=4); dot, = axes[2].plot([], [], "o", ms=13, color=ORANGE, mec="white", mew=2)
axes[2].set(xlabel="shift right (px)", ylabel="test accuracy", ylim=(0, 1.02), title="accuracy on the whole test set")
plt.tight_layout()

def update(t):
    dx, p = dxs[t], frames_p[t]; pred = p.argmax()
    im.set_data(frames_x[t][0])
    for c, b in enumerate(bars):
        b.set_height(p[c]); b.set_color(GREEN if c == pred == true else RED if c == pred else (LAVENDER if c == true else GREY))
    axes[0].set_title(f"true digit {true}, shifted {dx:+d} px", color=NAVY)
    axes[1].set_title(f"MLP says: {pred}  ({p[pred]:.0%})", color=GREEN if pred == true else RED)
    dot.set_data([dx], [curve_acc[list(curve_dx).index(dx)]])
    return []

play(animation.FuncAnimation(fig, update, frames=len(dxs), interval=350))

A shift of 2–3 pixels, which no human would even notice, and the accuracy collapses. The MLP has no idea that a pattern
means the same thing **wherever it appears**. We could show it shifted digits during training (*data augmentation*), but then
it has to learn every digit at every position separately: more data, more parameters, longer training.
In notebook 02 we repeat this experiment with a CNN, which has this knowledge built into its architecture.

---
# 4 · Experiment: does the MLP care where the pixels are?

We apply the **same fixed random shuffle** of the 784 pixel positions to *every* image (train and test).
For a human the digits become unrecognisable noise. What happens to the MLP?

In [ ]:
torch.manual_seed(42)
perm = torch.randperm(784)
shuffle_pixels = lambda X: X.reshape(len(X), -1)[:, perm].reshape(-1, 28, 28)
X_train_perm, X_test_perm = shuffle_pixels(X_train), shuffle_pixels(X_test)

fig, axes = plt.subplots(2, 8, figsize=(12, 3.2))
for k in range(8):
    axes[0, k].imshow(X_train[k], cmap="gray_r"); axes[1, k].imshow(X_train_perm[k], cmap="gray_r")
    axes[0, k].set_title(f"label {y_train[k].item()}")
for ax in axes.ravel(): ax.axis("off")
axes[0, 0].text(-6, 14, "original", ha="right", va="center", color=NAVY); axes[1, 0].text(-6, 14, "pixels\nshuffled", ha="right", va="center", color=ORANGE)
plt.tight_layout(); plt.show()

In [ ]:
torch.manual_seed(0)
mlp_perm = build_mlp()
hist_mlp_perm = fit(mlp_perm, *make_loaders(X_train_perm, y_train, X_test_perm, y_test), epochs=5)
print(f"\nMLP on original images: {hist_mlp['test_acc'][-1]:.2%}")
print(f"MLP on shuffled images: {hist_mlp_perm['test_acc'][-1]:.2%}")

**The MLP doesn't care!** It gets (almost) the same accuracy on shuffled images.

A fully-connected layer treats its 784 inputs as an *unordered list*: shuffling the inputs just shuffles the rows of
the first weight matrix, so it can learn the shuffled version equally well.

That sounds like a feature, but it is a **weakness**: the MLP does not use the most important prior knowledge we have about images:
* nearby pixels belong together (**locality**),
* a pattern (an edge, a hull, a "loop" of a 6) means the same thing wherever it appears (**translation**, see Section 3).

It has to learn everything from scratch, separately for every position. On 28 × 28 digits we can afford that. On real photos?

---
# 5 · From 28 × 28 to real photos: does the MLP scale?

MNIST images are tiny: 784 numbers. Our trawler at the standard input resolution of 224 × 224 × 3 has **150,528**, i.e.
192 × more. In an MLP every input is connected to every neuron of the first hidden layer, so the first layer grows with
the **number of pixels**. Three ways to size the hidden layer:
* keep **256 neurons** like our MNIST MLP (a bottleneck: 150,528 numbers squeezed into 256 right away),
* **1000 neurons**, the number on the lecture slide,
* keep the **same shape** as our MNIST MLP (784 → 256, roughly one hidden neuron per three inputs).

Training needs about **16 bytes per parameter**: the weight, its gradient, and Adam's two running averages (4 bytes each).

In [ ]:
def mlp_param_count(n_in, hidden):
    return n_in * hidden + hidden + hidden * 128 + 128 + 128 * 10 + 10        # n_in → hidden → 128 → 10

def available_memory():
    """Free memory for training on this machine in bytes: GPU memory, or RAM on a CPU runtime."""
    if device.type == "cuda": return torch.cuda.mem_get_info()[0]
    import psutil; return psutil.virtual_memory().available

BYTES_PER_PARAM = 16
SIZES = {"28×28×1\n(MNIST)": 28 * 28, "64×64×3": 64 * 64 * 3, "128×128×3": 128 * 128 * 3, "224×224×3\n(a photo)": 224 * 224 * 3}
WIDTHS = {"hidden = 256": lambda n: 256, "hidden = 1000": lambda n: 1000, "same shape as our MLP (inputs / 3)": lambda n: n // 3}

print(f"{'input':<11}" + "".join(f"{w.split(' (')[0]:>36}" for w in WIDTHS))
for name, n in SIZES.items():
    cols = [f"{mlp_param_count(n, f(n)):,} params ({mlp_param_count(n, f(n)) * BYTES_PER_PARAM / 1e9:.1f} GB)" for f in WIDTHS.values()]
    print(f"{name.splitlines()[0]:<11}" + "".join(f"{c:>36}" for c in cols))

mem = available_memory()
fig, ax = plt.subplots(figsize=(9, 4))
for (w, f), c in zip(WIDTHS.items(), [GREEN, ORANGE, RED]):
    ax.plot(range(len(SIZES)), [mlp_param_count(n, f(n)) * BYTES_PER_PARAM / 1e9 for n in SIZES.values()], "-o", color=c, lw=2.5, label=w)
ax.axhline(mem / 1e9, color=NAVY, ls="--", lw=1.5)
ax.text(-0.1, mem / 1e9 * 1.3, f"free {'GPU memory' if device.type == 'cuda' else 'RAM'} on this machine: {mem / 1e9:.0f} GB", color=NAVY)
ax.set(yscale="log", xticks=range(len(SIZES)), ylabel="memory needed for training (GB)",
       title="Memory needed to train an MLP:  inputs → hidden → 128 → 10")
ax.set_xticklabels(SIZES.keys()); ax.legend(loc="lower right", fontsize=9); plt.show()

Even when it fits, it is slow. Let's **measure** one training step (forward + backward + Adam update, a mini-batch of 64
random images) for the MNIST MLP and for MLPs on 224 × 224 × 3 inputs, and extrapolate to a full epoch:

In [ ]:
def sync():
    if device.type == "cuda": torch.cuda.synchronize()

def seconds_per_step(model, input_shape, batch=64, min_seconds=1.0):
    """Average wall-clock time of one training step on random data (at least 3 steps and 1 s, after a warm-up step)."""
    model.to(device); opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    x = torch.randn(batch, *input_shape, device=device); y = torch.randint(0, 10, (batch,), device=device)
    def step():
        loss = F.cross_entropy(model(x), y)
        opt.zero_grad(); loss.backward(); opt.step(); sync()
    step()                                                     # warm-up
    t0, n = time.time(), 0
    while n < 3 or time.time() - t0 < min_seconds:
        step(); n += 1
    return (time.time() - t0) / n

make_mlp = lambda n_in, hidden: nn.Sequential(nn.Flatten(), nn.Linear(n_in, hidden), nn.ReLU(),
                                             nn.Linear(hidden, 128), nn.ReLU(), nn.Linear(128, 10))
timings = {}
for name, shape, hidden in [("MNIST MLP   28×28×1,   hidden 256", (1, 28, 28), 256),
                            ("photo MLP  224×224×3,  hidden 256", (3, 224, 224), 256),
                            ("photo MLP  224×224×3,  hidden 1000", (3, 224, 224), 1000)]:
    n_params = mlp_param_count(int(np.prod(shape)), hidden)
    if n_params * BYTES_PER_PARAM > 0.6 * available_memory():
        print(f"{name}: skipped, does not fit into memory"); continue
    torch.manual_seed(0); model = make_mlp(int(np.prod(shape)), hidden)
    t = seconds_per_step(model, shape); timings[name] = t
    del model
    if device.type == "cuda": torch.cuda.empty_cache()
    epoch_60k, epoch_imagenet = t * 60_000 / 64, t * 1_281_167 / 64
    print(f"{name}: {n_params:>12,} params | {t * 1000:7.1f} ms per step | one epoch over 60k images: "
          f"{epoch_60k / 60:6.1f} min | over ImageNet (1.28 M images): {epoch_imagenet / 3600:5.1f} h")

base = timings.pop("MNIST MLP   28×28×1,   hidden 256")
for name, t in timings.items(): print(f"→ {name}: {t / base:.0f}× slower per step than the MNIST MLP")

And the "same shape as our MLP" version? It needs billions of parameters. Let's try to create just its **first layer**:

In [ ]:
n_in = 224 * 224 * 3; hidden = n_in // 3
n_params = mlp_param_count(n_in, hidden)
print(f"MLP {n_in:,} → {hidden:,} → 128 → 10: {n_params / 1e9:.1f} billion parameters")
print(f"  weights alone: {n_params * 4 / 1e9:.0f} GB · training with Adam: {n_params * BYTES_PER_PARAM / 1e9:.0f} GB · "
      f"free on this machine: {available_memory() / 1e9:.0f} GB")
if device.type == "cuda":
    try:
        first_layer = nn.Linear(n_in, hidden, device=device)
        print("…it fitted?! You have a very big GPU.")
        del first_layer
    except torch.cuda.OutOfMemoryError as e:
        print("\n❌ torch.cuda.OutOfMemoryError:", ". ".join(str(e).split(". ")[:2]))
    torch.cuda.empty_cache()
else:
    print("\n❌ Not even trying on a CPU runtime: allocating this much RAM would crash the notebook.")

**MLPs do not scale to real images.**
* The number of weights grows with the number of pixels: 192 × more inputs means 192 × more first-layer weights.
* Training gets **slower** by orders of magnitude, even for a modest hidden layer (see the measurement above).
* An MLP with the same shape as our MNIST network **does not even fit into memory**.
* And even if it did: billions of parameters, each tied to one pixel position, would need an enormous amount of data
  to learn every pattern at every position (Section 3).

We need a layer that **reuses** the same few weights at every position of the image: the **convolution**.

**Next →** [02 · Convolutional Neural Networks](https://colab.research.google.com/github/TorbenGl/visionseminar/blob/main/notebooks/02_convolutional_neural_networks.ipynb):
networks built for images. We will repeat the shift and the shuffle experiments there.

---
# Summary
1. To a computer an image is a $H\times W\times C$ array of numbers, and raw pixel distance says little about content.
2. Core CV tasks: **classification**, **detection**, **segmentation**, **depth estimation**; all need good spatial features.
3. An MLP on flattened MNIST reaches ~97.5 % test accuracy with 235 k parameters.
4. But it is **fragile**: its weights are glued to pixel positions, so shifting a digit by 2–3 pixels breaks it.
5. It is **blind to spatial structure**: shuffling the pixels doesn't hurt it.
6. It **does not scale**: at 224 × 224 × 3 the first layer alone has tens of millions to billions of weights, training slows
   down by orders of magnitude, and the full-size version does not fit into memory. CNNs fix all three.